## Baseline Model

In [4]:
import pandas as pd

train_df = pd.read_csv(".../data/processed/train.csv")
val_df = pd.read_csv(".../data/processed/val.csv")

test_paraphraser = pd.read_csv(
    ".../data/processed/test_paraphraser.csv"
)

test_rupaws = pd.read_csv(
    ".../data/processed/test_rupaws.csv"
)

print(train_df.shape)
print(val_df.shape)

(10006, 3)
(1766, 3)


In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "ai-forever/ruBert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
tokenizer(
    "Это первое предложение.",
    "Это второе предложение.",
    truncation=True,
    max_length=128
)

{'input_ids': [101, 736, 6083, 6309, 126, 102, 736, 8725, 6309, 126, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [7]:
import torch
from torch.utils.data import Dataset

class ParaphraseDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.sentence1 = df["sentence1"].tolist()
        self.sentence2 = df["sentence2"].tolist()
        self.labels = df["label"].tolist()

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.sentence1[idx],
            self.sentence2[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }

In [8]:
from torch.utils.data import DataLoader

train_dataset = ParaphraseDataset(
    train_df,
    tokenizer
)

val_dataset = ParaphraseDataset(
    val_df,
    tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)


In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params we

In [11]:
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print(device)

model = model.to(device)

mps


In [12]:
batch = next(iter(train_loader))

batch = {
    key: value.to(device)
    for key, value in batch.items()
}

outputs = model(**batch)

print(outputs.loss)
print(outputs.logits.shape)

tensor(0.6573, device='mps:0', grad_fn=<NllLossBackward0>)
torch.Size([16, 2])


In [14]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [15]:
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

EPOCHS = 3

best_f1 = 0.0
best_state = None

for epoch in range(EPOCHS):
    # =========================
    # TRAIN
    # =========================
    model.train()

    train_loss = 0.0
    train_preds = []
    train_labels = []

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS} [train]"
    )

    for batch in progress:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = outputs.logits.argmax(dim=1)

        train_preds.extend(preds.detach().cpu().numpy())
        train_labels.extend(
            batch["labels"].detach().cpu().numpy()
        )

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss /= len(train_loader)

    train_f1 = f1_score(
        train_labels,
        train_preds,
        average="binary"
    )

    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_loss = 0.0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        progress = tqdm(
            val_loader,
            desc=f"Epoch {epoch + 1}/{EPOCHS} [val]"
        )

        for batch in progress:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            val_loss += outputs.loss.item()

            preds = outputs.logits.argmax(dim=1)

            val_preds.extend(
                preds.cpu().numpy()
            )

            val_labels.extend(
                batch["labels"].cpu().numpy()
            )

    val_loss /= len(val_loader)

    val_f1 = f1_score(
        val_labels,
        val_preds,
        average="binary"
    )

    print(
        f"\nEpoch {epoch + 1}: "
        f"train_loss={train_loss:.4f}, "
        f"train_f1={train_f1:.4f}, "
        f"val_loss={val_loss:.4f}, "
        f"val_f1={val_f1:.4f}"
    )

    # сохраняем лучшую модель
    if val_f1 > best_f1:
        best_f1 = val_f1

        best_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

        print(f"New best F1: {best_f1:.4f}")

Epoch 1/3 [train]:   0%|          | 0/626 [00:00<?, ?it/s]

Epoch 1/3 [val]:   0%|          | 0/111 [00:00<?, ?it/s]


Epoch 1: train_loss=0.5071, train_f1=0.6136, val_loss=0.5197, val_f1=0.7128
New best F1: 0.7128


Epoch 2/3 [train]:   0%|          | 0/626 [00:00<?, ?it/s]

Epoch 2/3 [val]:   0%|          | 0/111 [00:00<?, ?it/s]


Epoch 2: train_loss=0.3353, train_f1=0.7934, val_loss=0.4158, val_f1=0.7696
New best F1: 0.7696


Epoch 3/3 [train]:   0%|          | 0/626 [00:00<?, ?it/s]

Epoch 3/3 [val]:   0%|          | 0/111 [00:00<?, ?it/s]


Epoch 3: train_loss=0.1879, train_f1=0.9020, val_loss=0.4592, val_f1=0.7846
New best F1: 0.7846


In [16]:
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)
from torch.utils.data import DataLoader


def evaluate_model(model, df, tokenizer, batch_size=16):
    dataset = ParaphraseDataset(
        df,
        tokenizer,
        max_length=128
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model.eval()

    predictions = []
    labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluation"):
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            preds = outputs.logits.argmax(dim=1)

            predictions.extend(
                preds.cpu().numpy()
            )

            labels.extend(
                batch["labels"].cpu().numpy()
            )

    f1 = f1_score(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1:       {f1:.4f}")
    print()

    print(
        classification_report(
            labels,
            predictions,
            digits=4
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(labels, predictions))

    return {
        "f1": f1,
        "accuracy": accuracy,
        "predictions": predictions,
        "labels": labels
    }

In [17]:
paraphraser_results = evaluate_model(
    model,
    test_paraphraser,
    tokenizer
)

Evaluation:   0%|          | 0/72 [00:00<?, ?it/s]

Accuracy: 0.9066
F1:       0.8626

              precision    recall  f1-score   support

           0     0.9487    0.9106    0.9293       772
           1     0.8296    0.8984    0.8626       374

    accuracy                         0.9066      1146
   macro avg     0.8892    0.9045    0.8960      1146
weighted avg     0.9099    0.9066    0.9075      1146

Confusion matrix:
[[703  69]
 [ 38 336]]


In [18]:
rupaws_results = evaluate_model(
    model,
    test_rupaws,
    tokenizer
)

Evaluation:   0%|          | 0/105 [00:00<?, ?it/s]

Accuracy: 0.7400
F1:       0.6877

              precision    recall  f1-score   support

           0     0.8170    0.7412    0.7773      1024
           1     0.6438    0.7381    0.6877       649

    accuracy                         0.7400      1673
   macro avg     0.7304    0.7396    0.7325      1673
weighted avg     0.7498    0.7400    0.7425      1673

Confusion matrix:
[[759 265]
 [170 479]]


## Lexical Overlap

In [19]:
import re

def tokenize_words(text):
    return set(
        re.findall(
            r"[а-яёa-z0-9]+",
            str(text).lower()
        )
    )


def lexical_overlap(sentence1, sentence2):
    tokens1 = tokenize_words(sentence1)
    tokens2 = tokenize_words(sentence2)

    if not tokens1 and not tokens2:
        return 1.0

    if not tokens1 or not tokens2:
        return 0.0

    return len(tokens1 & tokens2) / len(tokens1 | tokens2)

In [21]:
rupaws_analysis = test_rupaws.copy()

rupaws_analysis["overlap"] = rupaws_analysis.apply(
    lambda row: lexical_overlap(
        row["sentence1"],
        row["sentence2"]
    ),
    axis=1
)

rupaws_analysis["overlap"].describe()

count    1673.000000
mean        0.599848
std         0.208772
min         0.071429
25%         0.448276
50%         0.583333
75%         0.739130
max         1.000000
Name: overlap, dtype: float64

In [22]:
rupaws_analysis["prediction"] = rupaws_results["predictions"]
rupaws_analysis["true_label"] = rupaws_results["labels"]

In [23]:
rupaws_analysis[
    ["sentence1", "sentence2", "label", "overlap", "prediction"]
].head()

,sentence1,sentence2,label,overlap,prediction
0,Каковы были основные эффекты землетрясения Кам...,Каковы были основные последствия землетрясения...,0,0.700000,0
1,Как сделать мой новый номер телефона в качеств...,Как я сделаю свой старый телефон администратор...,0,0.236842,0
2,Почему страны не могут себе позволить продукци...,"Почему страны, не позволяющие себе высококачес...",1,0.379310,0
3,Нравятся ли мексиканским женщинам мужчины из В...,Нравятся ли женщинам Восточной Азии мексиканск...,0,0.750000,0
4,Какие правила вождения в Джорджии против Мисси...,Каковы правила вождения в штате Миссисипи прот...,1,0.666667,0


In [24]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]

rupaws_analysis["overlap_bin"] = pd.cut(
    rupaws_analysis["overlap"],
    bins=bins,
    include_lowest=True
)

In [25]:
rupaws_analysis["overlap_bin"].value_counts().sort_index()

overlap_bin
(-0.001, 0.2]     38
(0.2, 0.4]       259
(0.4, 0.6]       600
(0.6, 0.8]       476
(0.8, 1.0]       300
Name: count, dtype: int64

In [26]:
from sklearn.metrics import f1_score

overlap_results = []

for interval, group in rupaws_analysis.groupby(
    "overlap_bin",
    observed=True
):
    f1 = f1_score(
        group["true_label"],
        group["prediction"],
        average="binary"
    )

    overlap_results.append({
        "overlap_bin": str(interval),
        "n": len(group),
        "f1": f1
    })

overlap_results = pd.DataFrame(overlap_results)

overlap_results

,overlap_bin,n,f1
0,"(-0.001, 0.2]",38,0.600000
1,"(0.2, 0.4]",259,0.552632
2,"(0.4, 0.6]",600,0.573394
3,"(0.6, 0.8]",476,0.714628
4,"(0.8, 1.0]",300,0.853261


In [28]:
pd.set_option("display.max_colwidth", None)

hard_negatives = rupaws_analysis[
    (rupaws_analysis["true_label"] == 0) &
    (rupaws_analysis["prediction"] == 1)
].copy()

hard_negatives = hard_negatives.sort_values(
    "overlap",
    ascending=False
)

print("Количество hard negatives:", len(hard_negatives))

hard_negatives[
    ["sentence1", "sentence2", "overlap"]
].head(20)

Количество hard negatives: 265


,sentence1,sentence2,overlap
36,"Что более распространено, либеральный демократ или консервативный республиканец? Почему так?","Что более распространено, консервативный демократ или либеральный республиканец? Почему так?",1.000000
582,"Благодаря нетрадиционным аранжировкам и уникальным композициям, группа моментально выделилась как на кампусе, так и в центре Кливленда.","Благодаря уникальным аранжировкам и нетрадиционным композициям, группа моментально выделилась как на кампусе, так и в центре Кливленда.",1.000000
1156,"Трамвайная линия была построена в 1913 году, расширена в 1923 году и была заброшена в 1983 году.","Трамвайная линия была построена в 1913 году, и была заброшена в 1923 и расширена в 1983 году.",1.000000
1438,"Первоначальное здание было переделано в 1967 году, позднее было построено в 1999 году для размещения как библиотеки, так и городской ратуши.","Первоначальное здание было построено в 1967 году, позднее было переделано в 1999 году для размещения как библиотеки, так и городской ратуши.",1.000000
1153,"Средний возраст жителей Касл Пойнт (Castle Point) на момент переписи 2011 года составлял 45 лет, по сравнению со средним национальным показателем 39 лет и средним региональным показателем 40 лет.","Средний возраст жителей Касл Пойнт (Castle Point) на момент переписи 2011 года составлял 45 лет, по сравнению со средним региональным показателем 39 лет и средним национальным показателем 40 лет.",1.000000
534,"(В этом случае теряются как длинные гласные, так и промежуточные согласные.)","(В этом случае теряются как промежуточные гласные, так и длинные согласные).",1.000000
568,"Артур Врэйли, 3-й барон Уэсли (17 июня 1824 - 28 декабря 1910), был либеральным ровесником и британским политиком.","Артур Врэйли, 3-й барон Уэсли (17 июня 1824 - 28 декабря 1910), был британским ровесником и либеральным политиком.",1.000000
1107,К концу лета 1944 года войска 3-го Белорусского фронта и 31-й армии подошли к границам Восточной Пруссии.,К концу лета 1944 года войска 31-го Белорусского фронта и 3-й армии подошли к границам Восточной Пруссии.,1.000000
1427,В период с 1885 по 1978 год следующие суда использовались в качестве паромов от Мальты до Гоцо:,Следующие суда использовались в качестве паромов от Гоцо до Мальты в период с 1885 по 1978 год:,1.000000
1049,"Замок был преобразован дважды: в 15 веке в первый раз и в 19 веке после того, как он был частично разрушен.","Замок был преобразован дважды: в 19 веке в первый раз и в 15 веке после того, как он был частично разрушен.",1.000000


## Hard Negative Mining

In [29]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

def predict_model(model, df, tokenizer, batch_size=16):
    dataset = ParaphraseDataset(df, tokenizer, max_length=128)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.eval()

    predictions = []
    probabilities = []
    labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Prediction"):
            labels_batch = batch["labels"].numpy()

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            probs = torch.softmax(outputs.logits, dim=1)
            preds = probs.argmax(dim=1)

            predictions.extend(preds.cpu().numpy())
            probabilities.extend(probs[:, 1].cpu().numpy())
            labels.extend(labels_batch)

    return {
        "predictions": np.array(predictions),
        "probabilities": np.array(probabilities),
        "labels": np.array(labels)
    }

In [30]:
val_results = predict_model(
    model,
    val_df,
    tokenizer
)

Prediction:   0%|          | 0/111 [00:00<?, ?it/s]

In [31]:
val_analysis = val_df.copy()

val_analysis["prediction"] = val_results["predictions"]
val_analysis["probability"] = val_results["probabilities"]
val_analysis["true_label"] = val_results["labels"]

In [32]:
val_analysis["overlap"] = val_analysis.apply(
    lambda row: lexical_overlap(
        row["sentence1"],
        row["sentence2"]
    ),
    axis=1
)

In [33]:
hard_negatives = val_analysis[
    (val_analysis["true_label"] == 0) &
    (val_analysis["prediction"] == 1)
].copy()

In [34]:
print("Hard negatives:", len(hard_negatives))

Hard negatives: 195


In [35]:
hard_negatives = hard_negatives.sort_values(
    "overlap",
    ascending=False
)

hard_negatives[
    ["sentence1", "sentence2", "overlap", "probability"]
].head(30)

,sentence1,sentence2,overlap,probability
99,Приорий был расформирован в 1199 году и основан в 1321 году.,Приорий был основан в 1199 году и расформирован в 1321 году.,1.000000,0.969161
940,Он переехал в Оттаву в 1891 году и обосновался в штате Иллинойс.,Он переехал в Оттаву и обосновался в штате Иллинойс в 1891 году.,1.000000,0.997642
1745,"Какие продукты питания люди считают нездоровыми, но на самом деле полезными?","Какие продукты питания люди считают полезными, но на самом деле нездоровыми?",1.000000,0.839146
98,"Выборы 2016 года в США: Поддержат ли сторонники Берни Сандерса Хиллари Клинтон, если он выиграет номинацию?","Выборы 2016 года в США: Поддержат ли сторонники Хиллари Клинтон Берни Сандерса, если он выиграет номинацию?",1.000000,0.753955
1039,"Во время возвращения в атмосферу космический аппарат был переведен на баллистическую траекторию, в результате чего он приземлился к западу от Аркалыка, примерно к северо-востоку от предполагаемой площадки посадки в Казахстане.","Во время возвращения в атмосферу космический аппарат был переведен на баллистическую траекторию, в результате чего он приземлился примерно к северо-востоку от Аркалыка, к западу от предполагаемой площадки посадки в Казахстане.",1.000000,0.983218
1014,"Деррик Уэйн Фазьер (Derrick Wayne Feszier) (родился 28 апреля 1977 в Далласе, штат Техас, 31 августа 2006 в Хантсвилле, Техас) был американцем, осужденным за убийство.","Деррик Уэйн Фазьер (Derrick Wayne Feszier) (28 апреля 1977 в Хантсвилле, Техас -- 31 августа 2006 в Далласе, штат Техас) был американцем, осужденным за убийство.",0.956522,0.736827
139,"Партнеры по импорту: Германия - 6%, Польша - 9,6%, Китай - 7,5%, Италия - 6,3%, Нидерланды - 5,3%, Словакия - 4,1% (2016 год).","Партнеры по импорту: Германия - 6%, Польша - 9,6%, Китай - 7,5%, Словакия - 6,3%, Нидерланды - 5,3%, Италия - 4,1% (2016)",0.944444,0.538794
1486,"Артур Врли, 3-й барон Врэтли (17 июня 1824 - 28 декабря 1910), был либеральным ровесником и британским политиком.","Артур Врэтли, 3-й барон Врэтли (17 июня 1824 -- 28 декабря 1910), был британским ровесником и либеральным политиком.",0.944444,0.732122
1320,"Shine, второй альбом группы, выпущен в промежутке между двумя общими датами и записан в 2009 году.","Shine, второй альбом группы, был записан в промежутке между двумя общими датами и выпущен в 2009 году.",0.937500,0.994528
388,"Маргот А. Тиен / Margot A. Thien (род. 29 декабря 1971 года в Сан-Диего, Калифорния) - олимпийский чемпион по синхронному плаванию и американский чемпион.","Маргот А. Тиен / Margot A. Thien (род. 29 декабря 1971 года в Сан-Диего, Калифорния) - олимпийский чемпион по синхронному плаванию и американский мастер спорта.",0.916667,0.743051


In [37]:
for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    print(
        f"overlap >= {threshold}: "
        f"{(hard_negatives['overlap'] >= threshold).sum()}"
    )

overlap >= 0.5: 121
overlap >= 0.6: 80
overlap >= 0.7: 52
overlap >= 0.8: 31
overlap >= 0.9: 16


In [47]:
hard_train = hard_negatives[
    hard_negatives["overlap"] >= 0.5
].copy()

hard_train = hard_train[
    ["sentence1", "sentence2", "true_label"]
].rename(columns={"true_label": "label"})

print("Hard negatives:", len(hard_train))
print(hard_train["label"].value_counts())

Hard negatives: 121
label
0    121
Name: count, dtype: int64


## Improved RuBert

In [48]:
train_hard = pd.concat(
    [train_df, hard_train],
    ignore_index=True
)

print("Original train:", len(train_df))
print("Hard negatives:", len(hard_train))
print("New train:", len(train_hard))

Original train: 10006
Hard negatives: 121
New train: 10127


In [49]:
train_hard_dataset = ParaphraseDataset(
    train_hard,
    tokenizer,
    max_length=128
)

val_dataset = ParaphraseDataset(
    val_df,
    tokenizer,
    max_length=128
)

train_hard_loader = DataLoader(
    train_hard_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

In [50]:
from transformers import AutoModelForSequenceClassification

model_hard = AutoModelForSequenceClassification.from_pretrained(
    "ai-forever/ruBert-base",
    num_labels=2
)

model_hard = model_hard.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params we

In [51]:
from torch.optim import AdamW

optimizer = AdamW(
    model_hard.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [52]:
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import torch

n_epochs = 3

best_val_f1 = 0.0

for epoch in range(n_epochs):
    # -------------------
    # Training
    # -------------------
    model_hard.train()

    train_losses = []
    train_preds = []
    train_labels = []

    progress_bar = tqdm(
        train_hard_loader,
        desc=f"Epoch {epoch + 1}/{n_epochs}"
    )

    for batch in progress_bar:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model_hard(**batch)

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)

        train_losses.append(loss.item())
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels.extend(batch["labels"].detach().cpu().numpy())

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = sum(train_losses) / len(train_losses)
    train_f1 = f1_score(
        train_labels,
        train_preds,
        average="binary"
    )

    # -------------------
    # Validation
    # -------------------
    model_hard.eval()

    val_losses = []
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in tqdm(
            val_loader,
            desc="Validation"
        ):
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model_hard(**batch)

            val_losses.append(outputs.loss.item())

            preds = outputs.logits.argmax(dim=1)

            val_preds.extend(
                preds.cpu().numpy()
            )
            val_labels.extend(
                batch["labels"].cpu().numpy()
            )

    val_loss = sum(val_losses) / len(val_losses)

    val_f1 = f1_score(
        val_labels,
        val_preds,
        average="binary"
    )

    print(
        f"\nEpoch {epoch + 1}: "
        f"train_loss={train_loss:.4f}, "
        f"train_f1={train_f1:.4f}, "
        f"val_loss={val_loss:.4f}, "
        f"val_f1={val_f1:.4f}"
    )

    # -------------------
    # Save best model
    # -------------------
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1

        torch.save(
            model_hard.state_dict(),
            "rubert_hard_negatives_best.pt"
        )

        print(f"New best F1: {val_f1:.4f}")

Epoch 1/3:   0%|          | 0/633 [00:00<?, ?it/s]

Validation:   0%|          | 0/111 [00:00<?, ?it/s]


Epoch 1: train_loss=0.5060, train_f1=0.6112, val_loss=0.4218, val_f1=0.7535
New best F1: 0.7535


Epoch 2/3:   0%|          | 0/633 [00:00<?, ?it/s]

Validation:   0%|          | 0/111 [00:00<?, ?it/s]


Epoch 2: train_loss=0.3344, train_f1=0.7941, val_loss=0.3276, val_f1=0.8024
New best F1: 0.8024


Epoch 3/3:   0%|          | 0/633 [00:00<?, ?it/s]

Validation:   0%|          | 0/111 [00:00<?, ?it/s]


Epoch 3: train_loss=0.1871, train_f1=0.9027, val_loss=0.3052, val_f1=0.8434
New best F1: 0.8434


In [53]:
paraphraser_hard_results = evaluate_model(
    model_hard,
    test_paraphraser,
    tokenizer
)

Evaluation:   0%|          | 0/72 [00:00<?, ?it/s]

Accuracy: 0.9180
F1:       0.8760

              precision    recall  f1-score   support

           0     0.9449    0.9326    0.9387       772
           1     0.8646    0.8877    0.8760       374

    accuracy                         0.9180      1146
   macro avg     0.9047    0.9102    0.9074      1146
weighted avg     0.9187    0.9180    0.9182      1146

Confusion matrix:
[[720  52]
 [ 42 332]]


In [54]:
rupaws_hard_results = evaluate_model(
    model_hard,
    test_rupaws,
    tokenizer
)

Evaluation:   0%|          | 0/105 [00:00<?, ?it/s]

Accuracy: 0.7693
F1:       0.7191

              precision    recall  f1-score   support

           0     0.8365    0.7744    0.8043      1024
           1     0.6814    0.7612    0.7191       649

    accuracy                         0.7693      1673
   macro avg     0.7589    0.7678    0.7617      1673
weighted avg     0.7763    0.7693    0.7712      1673

Confusion matrix:
[[793 231]
 [155 494]]


In [56]:
rupaws_hard_analysis = test_rupaws.copy()

rupaws_hard_analysis["overlap"] = rupaws_hard_analysis.apply(
    lambda row: lexical_overlap(
        row["sentence1"],
        row["sentence2"]
    ),
    axis=1
)

rupaws_hard_analysis["prediction"] = rupaws_hard_results["predictions"]
rupaws_hard_analysis["true_label"] = rupaws_hard_results["labels"]

In [57]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]

rupaws_analysis["overlap_bin"] = pd.cut(
    rupaws_analysis["overlap"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

rupaws_hard_analysis["overlap_bin"] = pd.cut(
    rupaws_hard_analysis["overlap"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

rows = []

for bucket in labels:
    base = rupaws_analysis[rupaws_analysis["overlap_bin"] == bucket]
    hard = rupaws_hard_analysis[rupaws_hard_analysis["overlap_bin"] == bucket]

    rows.append({
        "overlap": bucket,
        "n": len(base),
        "baseline_f1": f1_score(
            base["true_label"],
            base["prediction"],
            zero_division=0
        ),
        "hard_negative_f1": f1_score(
            hard["true_label"],
            hard["prediction"],
            zero_division=0
        )
    })

comparison = pd.DataFrame(rows)
comparison["delta"] = (
    comparison["hard_negative_f1"]
    - comparison["baseline_f1"]
)

comparison

,overlap,n,baseline_f1,hard_negative_f1,delta
0,0-0.2,38,0.600000,0.692308,0.092308
1,0.2-0.4,259,0.552632,0.571429,0.018797
2,0.4-0.6,600,0.573394,0.626794,0.053400
3,0.6-0.8,476,0.714628,0.723192,0.008564
4,0.8-1.0,300,0.853261,0.885870,0.032609
